# Transient chaos — phase-space Shannon entropy

Copy of `analysis/no_pump_frequency/phase_space_shannon_entropy copy.ipynb` (09/10/2026), with the parameters of the other
`poster_chicago_2026` notebooks:

- $\omega_a = 25.338776456203686$, fixed. The original drew it at random: `3.85*(rand(1)[1]/5+0.9)*2π`.
- $\kappa_b = \pi/10.4$, in the $-2\kappa_b\dot\phi_b$ convention. The original used $2/10.4$.
- $\varepsilon_d = 10.12\,g_2$ in both simulation loops, through `ϵ_d_g2_ratio`. The original parameter cell used 10.12, but the loops used 10.
- $\kappa_a = 0$, $\omega_d = \omega_b$, $\omega_p = 2\omega_a - \omega_d = 0$, same $\varphi_a, \varphi_b, E_J$ as before.
- The pump amplitude is called `ϵ_p` (it was `ϵ_0`). Unused variables (`Lamb_shift`, `κ_2`, the cell-level `ϵ_0`, `g2`, `ϵ_d`) are removed.

**12/10/2026:** $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$ (static pump), instead of $J_1(\varepsilon_p)E_J\varphi_a^2\varphi_b$, so $\varepsilon_d = 10.12\,g_2$ would be about twice as large as before at the same $\varepsilon_p$. There the chaos does not decay within /\kappa_b$. The ratio is therefore set to $\varepsilon_d/g_2 = 5.2$, the same physical drive as the former .12\,g_2^{J_1}$ at $\varepsilon_p = 0.802$ (/2\pi = 98.8$ MHz with the new definition).

The values $\varepsilon_p = 0.8$ and $0.802$ were chosen with the old $\kappa_b$. Times are in units of $1/\kappa_b$,
so $t_{max} = 2200/\kappa_b$ is about 1.6× shorter in ns than before.


In [ ]:
using ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

ϵ_d_g2_ratio = 5.2  # ϵ_d = ϵ_d_g2_ratio*g2. Same physical drive as the former 10.12 J1(ϵ_p)E_Jφ_a²φ_b at ϵ_p = 0.802 (10.12 J1/sin = 5.20), where the chaos is transient

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4 # 1/ns, convention φ̈_b = … − 2κ_b φ̇_b (db/dt = … − κ_b b for SystemATS_ladder)
κ_a = 0

In [ ]:
ϵ_p_list = [0.8]#range(start = 0.6044444444444445, stop = 0.802, length = 10);


t_max = 1500/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,10)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_p in ϵ_p_list
    g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b # static pump (ω_p = 0)
    ϵ_d=ϵ_d_g2_ratio*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Final_mode[(re_i,im_j)] = a
        end
    end
    Final_all[ϵ_p] = Final_mode
end

In [ ]:
for ϵ_p in ϵ_p_list

    g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b
    Final_mode = Final_all[ϵ_p]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        scatter!(p0,
                 real.(Final_mode[(re_i, im_j)]),
                 imag.(Final_mode[(re_i, im_j)]),
                 ms=1, label="", color=:black)
    end

    # Add labels and title
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          top_margin=5mm)

    display(p0)
end


In [ ]:
ϵ_p_list = [0.802]#range(start = 0.6044444444444445, stop = 0.802, length = 10);


t_max = 500/κ_b
t_min = 0.0
t_min_save = t_min
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, b_0)

environment_prepared = prepare(system,environment);
t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,10)
Final_all = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for ϵ_p in ϵ_p_list
    g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b # static pump (ω_p = 0)
    ϵ_d=ϵ_d_g2_ratio*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Final_mode = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Final_mode[(re_i,im_j)] = a
        end
    end
    Final_all[ϵ_p] = Final_mode
end

In [ ]:
dict = Final_all[ϵ_p_list[1]]

# Phase-space resolution
nx = 200
ny = 200

# Collect all points once to define bounds
all_points = reduce(vcat, values(dict))
x_all = real.(all_points)
y_all = imag.(all_points)

x_edges = range(minimum(x_all), maximum(x_all); length=nx+1)
y_edges = range(minimum(y_all), maximum(y_all); length=ny+1)

In [ ]:
using StatsBase

function shannon_entropy(weights)
    p = weights ./ sum(weights)
    p = p[p .> 0]                 # avoid log(0)
    return -sum(p .* log.(p))
end
group_size = 200
trajectories = collect(values(dict))
ntraj = length(trajectories)

# number of chunks (use minimum length to stay safe)
nchunks = minimum(length(traj) ÷ group_size for traj in trajectories)

entropy_per_chunk = zeros(Float64, nchunks, ntraj)

for (j, traj) in enumerate(trajectories)
    for k in 1:nchunks
        i0 = (k-1)*group_size + 1
        i1 = k*group_size
        chunk = @view traj[i0:i1]

        h = fit(
            Histogram,
            (real.(chunk), imag.(chunk)),
            (x_edges, y_edges)
        )

        entropy_per_chunk[k, j] = shannon_entropy(h.weights)
    end
end

nbins_entropy = 500

Hmin = minimum(entropy_per_chunk)
Hmax = maximum(entropy_per_chunk)

entropy_bins = range(Hmin, Hmax; length=nbins_entropy+1)
entropy_centers = (entropy_bins[1:end-1] .+ entropy_bins[2:end]) ./ 2

entropy_hist_matrix = zeros(Float64, nbins_entropy, nchunks)

for k in 1:nchunks
    h = fit(Histogram, entropy_per_chunk[k, :], entropy_bins)
    entropy_hist_matrix[:, k] .= h.weights
end

chunk_indices = (0:nchunks-1) .* group_size .+ 1   # index of first point in chunk
chunk_time_converted = chunk_indices*dt_save*κ_b

In [ ]:

heatmap(
    chunk_time_converted,
    entropy_centers,
    entropy_hist_matrix;
    cmap = :viridis,
    xlabel = L"\kappa_bt",
    ylabel = "Shannon entropy",
    colorbar = true,
    #ylims=(5,7)
)

In [ ]:
using EasyFit
H_thresh_chaos = 5.0   # example threshold (choose physically meaningful value)
proportion_chaos = [count(>=(H_thresh_chaos), entropy_per_chunk[k, :]) for k in 1:nchunks-5]./length(Re_a_list)./length(Im_a_list)

H_thresh_reg = 5.0
proportion_reg = [count(<=(H_thresh_reg), entropy_per_chunk[k, :]) for k in 1:nchunks-5]./length(Re_a_list)./length(Im_a_list)

fit_exp_c = fitexp(chunk_time_converted[1:end-5],proportion_chaos)
fit_exp_r = fitexp(chunk_time_converted[1:end-5],proportion_reg)

A_chaos,B_chaos,C_chaos = fit_exp_c.a,fit_exp_c.b,fit_exp_c.c
A_reg,B_reg,C_reg = fit_exp_r.a,fit_exp_r.b,fit_exp_r.c

time_lis = 0:1:maximum(chunk_time_converted)
prop_chaos_lis =@. A_chaos*exp(-time_lis/B_chaos)+C_chaos
prop_reg_lis = @. A_reg*exp(-time_lis/B_reg)+C_reg;

In [ ]:
plot(
    chunk_time_converted[1:end-5]*2,
    proportion_chaos,
    lw = 2,
    label = "Chaotic traj."
)
plot!(
    time_lis*2,
    prop_chaos_lis,
    ls =:dash,
    label=""
)
plot!(
    chunk_time_converted[1:end-5]*2,
    proportion_reg,
    lw = 2,
    label = "Regular traj."
)
plot!(
    time_lis*2,
    prop_reg_lis,
    ls =:dash,
    label=""
)
plot!(
    xlabel = L"\kappa_bt",
    ylabel = "Fraction of trajectories",
#    ylims = (0.0, 1)
)

#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/transient_chaos_12_10_2026.pdf")
savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/article_chaos/figures_article/transient_chaos_12_10_2026.pdf")

## Lifetime of the transient chaos against $\varepsilon_p$ (13/10/2026, adaptive duration and finer sweep 16/10/2026)

Scan of $\varepsilon_p$ from 0.66 to 1.0 ($g_2/2\pi \approx 84$–$122$ MHz) at the same $\varepsilon_d = 5.2\,g_2$ ($\varepsilon_d/2g_2 = 2.6$), with the protocol of the cells above. All times are $\kappa_b t$ in the article convention ($\kappa_b/2\pi = 96$ MHz, i.e. twice the code $\kappa_b t$).

- **Escape time** of a trajectory: end of its last chaotic window ($H \ge H_c = 5$). Only trajectories chaotic in at least one window count. Trajectories not yet on a regular attractor at the end of the run ($H > H_{reg} = 2$ in the last window) are censored.
- **Survival** $S(t)$: fraction of these trajectories still chaotic after $t$. For a boundary crisis, $S(t) \approx e^{-t/\tau}$ and $\tau \propto (g_2 - g_{2,c})^{-\gamma}$ above the crisis.
- $\tau$ from an exponential fit of $S(t)$, cross-checked with the mean escape time of the uncensored trajectories (a lower bound when many are censored).
- **Adaptive duration**, for each $\varepsilon_p$ separately: run up to $\kappa_b t = 375/4 \approx 94$ and fit. Then double the duration, continuing the same trajectories from their final state, and fit again. Stop when $\tau$ is positive and within 10 % of the previous one, or at $\kappa_b t = 3000$. The durations are rounded to whole windows: 90, 180, 390, 749, 1498, 2996.
- If the exponential fit still fails at $\kappa_b t = 3000$ (no fit, or $\tau \le 0$), $S(t)$ is fitted by a constant instead: no measurable decay, $\kappa_b\tau > 3000$. Shown as triangles at $\kappa_b t_{max}$, and as a dashed line on the survival plot.
- **Finer sweep** (20 new values of $\varepsilon_p$) around the onset of the transient chaos, between two points of the coarse sweep:
  - $\varepsilon_{hi}$: the first $\varepsilon_p$ whose survival decays (no constant fit) with a mean escape time $\kappa_b\tau_{mean} < 100$;
  - $\varepsilon_{lo}$: the largest $\varepsilon_p$ below $\varepsilon_{hi}$ with a constant fit.
  - Constant fits are excluded for $\varepsilon_{hi}$ because their $\tau_{mean}$ comes from the few trajectories that escaped (30 at 86.4 MHz, from 1 trajectory out of 58).
- Resolution: one window is 200 memory periods, i.e. $30/\kappa_b$. Shorter lifetimes are not resolved.

Cells:
- Cell 14: the parameters and the functions (`adaptive_lifetime(ϵ_p)` for one $\varepsilon_p$, `plot_lifetimes(results)`).
- Cell 15: the coarse sweep, a few minutes on 10 threads. Cell 16 plots it.
- Cell 17: the finer sweep, a few minutes (about 7 s per $\varepsilon_p$ that runs up to 3000). Cell 18 plots both sweeps together.

**16/10/2026:** two fixes to the survival curve.
- The chaotic band ($H \approx 4.5$–$5.3$) flickers across $H_c$. A trajectory that stays chaotic has about 20 % of its windows below $H_c$.
- Before the fix, a trajectory counted as censored only if its last window was chaotic. The rest got the end of their last chaotic window as escape time, a few windows before $t_{max}$. $S(t)$ then dropped spuriously just before $t_{max}$, whatever $t_{max}$ was.
- Censored trajectories also had escape time $t_{max}$ instead of $\infty$, which forced $S(t_{max}) = 0$.

At 84.3 MHz all 54 chaotic trajectories are still chaotic at $\kappa_b t = 4000$. Before the fix, 8 to 13 of them were counted as escaped.

In [ ]:
# Lifetime of the transient chaos against the pump amplitude, at fixed ϵ_d = ϵ_d_g2_ratio*g2 (cell 2).
# Same protocol as the cells above: 10×10 grid of initial memory states, b(0) = 0, one sample per memory period,
# windows of M = 200 periods, Shannon entropy on a 200×200 partition of the bounding box of all sampled points
# (one partition per ϵ_p), chaotic window if H ≥ H_c, regular attractor if H ≤ H_reg.
# Times: κ_b here is the code rate (−2κ_b φ̇_b). In the article convention (κ_b/2π = 96 MHz), t_article = 2κ_b t.
# All the durations below (T_0_art, T_max_art, t_win, τ) are κ_b t in the article convention.
using StatsBase, EasyFit

ϵ_p_scan = collect(0.66:0.02:1.0)      # g2/2π ≈ 84–122 MHz
dt_save_scan = 2π/ω_a                   # Poincaré map of the memory
M_scan = 200
H_c_scan = 5.0
H_reg_scan = 2.0                        # the chaotic band is H ≈ 4.5–5.3 and flickers across H_c; regular attractors give H ≲ 2
n_bins_scan = 200
Re_a_scan = range(-5, 5, 10)
Im_a_scan = range(-4, 4, 10)

κ_art = 2κ_b                                   # article convention
T_win_art = M_scan*dt_save_scan*κ_art          # one window ≈ 30
T_0_art = 375.0/4                              # first duration (≈ 94, i.e. 3 windows), then doubled up to T_max_art
T_max_art = 6000.0
rtol_τ = 0.1                                   # τ converged if positive and within 10% of the τ of the previous duration

function window_entropies(trajs, M, nbins)
    pts = reduce(vcat, trajs)
    x_edges = range(minimum(real.(pts)), maximum(real.(pts)); length=nbins+1)
    y_edges = range(minimum(imag.(pts)), maximum(imag.(pts)); length=nbins+1)
    K = minimum(length.(trajs)) ÷ M
    H = zeros(K, length(trajs))
    for (j, traj) in enumerate(trajs), k in 1:K
        chunk = @view traj[(k-1)*M+1:k*M]
        w = fit(Histogram, (real.(chunk), imag.(chunk)), (x_edges, y_edges)).weights
        q = w[w .> 0] ./ M
        H[k, j] = -sum(q .* log.(q))
    end
    return H
end

# Continues every trajectory up to sample n_new (sample n at t = n*dt_save_scan; trajs[k] holds samples 0, 1, …).
# u[k] is the full-precision state [a, b] at the last stored sample, the stored samples are ComplexF32.
function extend!(trajs, u, drive_prepared, n_new)
    n_old = length(trajs[1]) - 1
    t_save = range(n_old*dt_save_scan, step=dt_save_scan, length=n_new-n_old+1)
    system_prepared = prepare(SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, 0.0))   # a_0, b_0 unused by the EOM
    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    Threads.@threads for k in eachindex(trajs)
        a, b, _ = default_solver(u[k], (t_save[1], t_save[end]), p, dt, t_save=t_save, equation=default_equation!)
        append!(trajs[k], ComplexF32.(a[2:end]))
        u[k] = [a[end], b[end]]
    end
end

# Escape time of a trajectory: end of its last chaotic window. Only trajectories chaotic in at least one window count.
# Censored (escape time Inf): not yet on a regular attractor at the end, H > H_reg in the last window.
# S(t): fraction of these trajectories with escape time > t. τ_fit: decay time of S(t) ≈ A e^{-t/τ} + C (EasyFit).
# τ_mean: mean escape time of the uncensored trajectories (a lower bound when many are censored).
function survival(H, t_win, H_c, H_reg)
    chaotic = H .>= H_c
    ever = findall(vec(any(chaotic, dims=1)))
    n_chaotic = length(ever)
    n_chaotic < 5 && return (S=fill(NaN, length(t_win)), τ_fit=NaN, τ_mean=NaN, n_chaotic=n_chaotic, n_censored=0)
    last_win = [findlast(chaotic[:, j]) for j in ever]
    censored = H[end, ever] .> H_reg
    t_esc = ifelse.(censored, Inf, t_win[last_win])
    S = [count(>(t), t_esc)/n_chaotic for t in t_win]
    τ_mean = any(.!censored) ? sum(t_esc[.!censored])/count(.!censored) : NaN
    τ_fit = NaN
    if !all(censored)
        try
            τ_fit = fitexp(t_win, S).b
        catch err
            @warn "fit failed" err
        end
    end
    return (S=S, τ_fit=τ_fit, τ_mean=τ_mean, n_chaotic=n_chaotic, n_censored=count(censored))
end

# Adaptive duration for one ϵ_p: run for T_0_art and fit S(t), then double the duration (continuing the same trajectories)
# and fit again. Stop when τ_fit is positive and within rtol_τ of the τ_fit of the previous duration, when there is no
# chaos (fewer than 5 chaotic trajectories), or after T_max_art. If the exponential fit still fails at T_max_art
# (no fit, or τ_fit ≤ 0), fit a constant instead: S(t) ≈ S_const, i.e. no measurable decay within T_max_art.
function adaptive_lifetime(ϵ_p)
    T_stages = [T_0_art*2^n for n in 0:floor(Int, log2(T_max_art/T_0_art))]
    g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b
    drive_prepared = prepare(DriveTwoModesBasics(ϵ_d_g2_ratio*g2, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase))
    a_0 = [x + 1im*y for x in Re_a_scan for y in Im_a_scan]
    u = [ComplexF64[a, 0.0] for a in a_0]
    trajs = [ComplexF32[a] for a in a_0]
    history = NamedTuple[]                                  # (T, τ_fit, τ_mean, n_chaotic, n_censored) at each duration
    H, s, converged = nothing, nothing, false
    for T in T_stages
        K = round(Int, T/T_win_art)
        extend!(trajs, u, drive_prepared, K*M_scan - 1)
        H = window_entropies(trajs, M_scan, n_bins_scan)
        s = survival(H, collect(1:K) .* T_win_art, H_c_scan, H_reg_scan)
        push!(history, (T=K*T_win_art, τ_fit=s.τ_fit, τ_mean=s.τ_mean, n_chaotic=s.n_chaotic, n_censored=s.n_censored))
        s.n_chaotic < 5 && break
        if length(history) ≥ 2
            τ_old = history[end-1].τ_fit
            converged = s.τ_fit > 0 && τ_old > 0 && abs(s.τ_fit - τ_old) ≤ rtol_τ*s.τ_fit
            converged && break
        end
    end
    S_const = !converged && s.n_chaotic ≥ 5 && !(s.τ_fit > 0) ? mean(s.S) : NaN   # least-squares constant
    println("ϵ_p = ", round(ϵ_p, digits=4), "  κ_b T = ", join([round(Int, h.T) for h in history], ", "),
            "  κ_b τ (fit) = ", join([round(h.τ_fit, digits=1) for h in history], ", "), converged ? "  converged" : "",
            isnan(S_const) ? "" : "  constant fit S = "*string(round(S_const, digits=3)))
    # H: (window × trajectory) entropies and surv: survival(…), both at the last duration
    return (ϵ_p=ϵ_p, g2=g2/2π*1e3, H=H, surv=s, history=history, converged=converged, S_const=S_const)
end

# Survival at the last duration of each ϵ_p (colour: g2), and τ against g2. Filled circles: converged τ_fit, open: not
# converged (τ_fit at T_max_art, or at the last duration before the chaos ran out). Triangles at κ_b t_max: constant fit
# of S(t), no measurable decay (κ_b τ > κ_b t_max), also shown dashed on the survival plot.
function plot_lifetimes(results)
    g2 = [r.g2 for r in results]
    τ_fit = [r.history[end].τ_fit for r in results]
    τ_mean = [r.history[end].τ_mean for r in results]
    conv = [r.converged for r in results]
    cst = [!isnan(r.S_const) for r in results]

    p_surv = plot(xlabel=L"\kappa_b t", ylabel="Surviving chaotic fraction", ylims=(1e-2, 1.1), legend=false,
                  colorbar=true, colorbar_title=L"g_2/2\pi"*" (MHz)", right_margin=8mm)
    for r in results
        r.surv.n_chaotic < 5 && continue
        t_win = collect(1:length(r.surv.S)) .* T_win_art
        plot!(p_surv, t_win, max.(r.surv.S, 1e-2), line_z=fill(r.g2, length(t_win)), color=:viridis, clims=extrema(g2))
        isnan(r.S_const) || plot!(p_surv, [0, t_win[end]], fill(r.S_const, 2), ls=:dash, line_z=fill(r.g2, 2),
                                  color=:viridis, clims=extrema(g2))
    end
    display(p_surv)

    p_tau = scatter(g2[conv], τ_fit[conv], xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\kappa_b\tau",
                    label="exponential fit of S(t), converged", legend=:topright, color=1)
    scatter!(p_tau, g2[.!conv .& .!cst], τ_fit[.!conv .& .!cst], markercolor=:white, markerstrokecolor=1, label="not converged")
    scatter!(p_tau, g2[cst], fill(T_max_art, count(cst)), marker=:utriangle, color=3, label="constant fit of S(t) (no decay)")
    scatter!(p_tau, g2, τ_mean, marker=:diamond, color=2, label="mean escape time (uncensored)")
    hline!(p_tau, [T_max_art], ls=:dash, color=:gray, label=L"\kappa_b t_{max}")
    plot!(ylim=(-.1*T_max_art,1.2*T_max_art))
    display(p_tau)

    for r in results
        h = r.history[end]
        println(rpad(round(r.g2, digits=1), 7), " MHz  κ_b T ", lpad(round(Int, h.T), 5), "  chaotic ", lpad(h.n_chaotic, 3),
                "  censored ", lpad(h.n_censored, 3), "  κ_b τ (fit) ", round(h.τ_fit, digits=1), "  κ_b τ (mean) ", round(h.τ_mean, digits=1),
                r.converged ? "" : "  not converged", isnan(r.S_const) ? "" : "  constant fit S = "*string(round(r.S_const, digits=3)))
    end
    return p_surv, p_tau
end

In [ ]:
# Coarse sweep in ϵ_p, adaptive duration for each ϵ_p (adaptive_lifetime, cell 14).
results = [adaptive_lifetime(ϵ_p) for ϵ_p in ϵ_p_scan]

In [ ]:
p_surv, p_tau = plot_lifetimes(results);

In [ ]:
# Finer sweep around the onset of the transient chaos, between two points of the coarse sweep:
# ϵ_hi, the first ϵ_p whose survival decays (no constant fit) with a mean escape time below τ_thresh, and
# ϵ_lo, the largest ϵ_p below ϵ_hi where S(t) needed a constant fit (no measurable decay within T_max_art).
# The constant fits are excluded for ϵ_hi because their τ_mean comes from the few trajectories that escaped.
τ_thresh = 100.0                                # κ_b τ, article convention
n_fine = 20                                     # new points, strictly between ϵ_lo and ϵ_hi

i_hi = findfirst(r -> isnan(r.S_const) && r.history[end].τ_mean < τ_thresh, results)
isnothing(i_hi) && error("no ϵ_p of the coarse sweep with κ_b τ_mean < $τ_thresh")
i_lo = findlast(r -> !isnan(r.S_const), results[1:i_hi-1])
isnothing(i_lo) && @warn "no constant fit below ϵ_p = $(results[i_hi].ϵ_p), the finer sweep starts at the first ϵ_p of the coarse sweep"
ϵ_lo, ϵ_hi = results[something(i_lo, 1)].ϵ_p, results[i_hi].ϵ_p
println("finer sweep between ϵ_p = $ϵ_lo (", round(results[something(i_lo, 1)].g2, digits=1), " MHz) and ϵ_p = $ϵ_hi (",
        round(results[i_hi].g2, digits=1), " MHz)")

ϵ_p_fine = collect(range(ϵ_lo, ϵ_hi, length=n_fine+2)[2:end-1])
results_fine = [adaptive_lifetime(ϵ_p) for ϵ_p in ϵ_p_fine]

In [ ]:
# Coarse and fine sweeps together.
results_all = sort(vcat(results, results_fine), by = r -> r.ϵ_p)
p_surv_all, p_tau_all = plot_lifetimes(results_all);

In [ ]:
conv = [r for r in results_all if r.converged]
ϵ_p_conv = [r.ϵ_p for r in conv]
τ_fit_conv = [r.history[end].τ_fit for r in conv]
plot(ϵ_p_conv, τ_fit_conv, marker=:circle, xlabel=L"\varepsilon_p", ylabel=L"\kappa_b\tau", label="converged")


In [ ]:
# Fit τ(ε_p) = A / (ε_p − ε_crit)^γ on the converged points.
# Taking logs gives log τ = log A − γ log(ε_p − ε_crit), linear in (log A, γ) for fixed ε_crit.
# ε_crit is scanned on a grid below the smallest converged ε_p, and the best residual is kept.
using Statistics

mask = τ_fit_conv .> 0                          # log needs positive τ
x = ϵ_p_conv[mask][1:15]
logy = log.(τ_fit_conv[mask])[1:15]

function powerlaw_fit(x, logy, ϵ_c)
    X = hcat(ones(length(x)), -log.(x .- ϵ_c))
    β = X \ logy
    return (res=sum(abs2, X*β - logy), A=exp(β[1]), γ=β[2])
end

ϵ_c_grid = range(0.0, minimum(x) - 1e-3, length=2000)
fits = [merge((ϵ_c=ϵ_c,), powerlaw_fit(x, logy, ϵ_c)) for ϵ_c in ϵ_c_grid]
best = fits[argmin([f.res for f in fits])]

println("ϵ_crit = ", round(best.ϵ_c, digits=4), "  A = ", round(best.A, digits=3),
        "  γ = ", round(best.γ, digits=3), "  rms(log τ) = ", round(sqrt(best.res/length(x)), digits=3))

ϵ_fit = range(best.ϵ_c + 1e-3, maximum(x)*1.02, length=300)
p_fit = scatter(x, τ_fit_conv[mask], label="converged", xlabel=L"\varepsilon_p", ylabel=L"\kappa_b\tau")
plot!(p_fit, ϵ_fit, best.A ./ (ϵ_fit .- best.ϵ_c).^best.γ, label="fit", lw=2, yscale=:log10)
display(p_fit)

